# Give Me Some Credit EDA

각 컬럼을 어떻게 전처리할지, 그리고 왜 그렇게 정했는지를 표와 그래프로 정리한다.

## 1. 준비

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from src.config import DATA_SET_PATH

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# 변수 정의
TARGET = "SeriousDlqin2yrs"
LATE_COLS = ["NumberOfTime30-59DaysPastDueNotWorse", "NumberOfTime60-89DaysPastDueNotWorse", "NumberOfTimes90DaysLate"]
BIN_DICT = {
    "age":                                  [0, 20, 30, 40, 50, 60, 70, 80, float("inf")],
    "RevolvingUtilizationOfUnsecuredLines": [0, 0.1, 0.3, 0.5, 0.7, 1, 2, 10, float("inf")],
    "DebtRatio":                            [0, 0.1, 0.3, 0.5, 0.7, 1, 2, 10, 300, float("inf")],
    "MonthlyIncome":                        [0, 1000, 2500, 5000, 7500, 10000, 20000, 50000, float("inf")],
}
COLOUM_MAPPING_DICT = {
    "SeriousDlqin2yrs":                     "90일 이상 연체 경험 여부 (타겟 변수)",
    "RevolvingUtilizationOfUnsecuredLines": "신용카드 및 개인 신용한도 대출의 총 잔액을 총 신용한도로 나눈 비율 (부동산 담보대출, 자동차 할부 같은 할부 대출은 제외)",
    "age":                                  "대출자 나이 (세)",
    "NumberOfTime30-59DaysPastDueNotWorse": "최근 2년간 30~59일 연체한 횟수 (그보다 심한 연체는 제외)",
    "DebtRatio":                            "월 부채 상환액, 위자료, 생활비를 월 총소득으로 나눈 비율",
    "MonthlyIncome":                        "월 소득",
    "NumberOfOpenCreditLinesAndLoans":      "보유 중인 대출(자동차 할부, 주택담보대출 등 할부 대출) 및 신용한도(신용카드 등) 개수",
    "NumberOfTimes90DaysLate":              "90일 이상 연체한 횟수",
    "NumberRealEstateLoansOrLines":         "주택담보대출 및 부동산 대출 개수 (주택담보 신용한도 포함)",
    "NumberOfTime60-89DaysPastDueNotWorse": "최근 2년간 60~89일 연체한 횟수 (그보다 심한 연체는 제외)",
    "NumberOfDependents":                   "본인을 제외한 부양가족 수 (배우자, 자녀 등)"
}
pd.DataFrame(COLOUM_MAPPING_DICT.items(), columns=["컬럼명", "설명"])

In [ ]:
# 데이터 로드
df = pd.read_csv(DATA_SET_PATH, index_col=0)
BASE_RATE = df[TARGET].mean() * 100
print(f"전체 {len(df):,}건, 전체 평균 부도율 {BASE_RATE:.2f}%")

In [ ]:
def show_default_rate(data, group, title):
    """그룹별 건수와 부도율을 표와 막대그래프로 보여준다."""
    table = data.groupby(group, observed=False)[TARGET].agg(["count", "mean"])
    table.columns = ["건수", "부도율(%)"]
    table["부도율(%)"] = (table["부도율(%)"] * 100).round(2)
    display(table)

    x = range(len(table))
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.bar(x, table["부도율(%)"], color="steelblue")
    ax.axhline(BASE_RATE, color="red", linestyle="--", label=f"전체 평균 부도율 {BASE_RATE:.1f}%")
    for i, (n, rate) in enumerate(zip(table["건수"], table["부도율(%)"])):
        ax.text(i, rate, f"{n:,}건", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(table.index.astype(str), rotation=30, ha="right")
    ax.set_title(title)
    ax.set_ylabel("부도율(%)")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 2. 전체 개요

In [ ]:
# 결측치 개수와 비율
pd.DataFrame({
    "결측치 개수": df.isna().sum(),
    "결측치 비율(%)": (df.isna().mean() * 100).round(2),
})

In [ ]:
# 기본 통계
df.describe(percentiles=[0.01, 0.5, 0.99]).T

In [ ]:
# 컬럼별 값 분포
for key, value in COLOUM_MAPPING_DICT.items():
    if key in BIN_DICT:
        col = pd.cut(df[key], bins=BIN_DICT[key], include_lowest=True)
    else:
        col = df[key]

    vc = col.value_counts().sort_index()
    table = pd.DataFrame({
        "개수": vc,
        "비율(%)": (vc / len(df) * 100).round(2)
    })
    display(Markdown(f"### {key}\n{value}"))
    display(table)

## 3. age

0세 고객이 있는지, 0 다음으로 작은 나이가 몇 살인지 확인한다.

**도표 3A.** age 값별 건수 (작은 값 5개)

In [ ]:
df["age"].value_counts().sort_index().head(5).to_frame("건수")

**결론: `age = 0`인 행은 삭제한다.**
- 0세는 대출을 받을 수 없으므로 명백한 오류다.
- 0세는 1건뿐이고 다음으로 작은 나이는 21세이기에 삭제해도 영향이 없다(도표 3A).

## 4. RevolvingUtilizationOfUnsecuredLines

RevolvingUtilizationOfUnsecuredLines는 원래 0~1 사이 값이어야 한다. 1을 넘는 값이 실제 고객인지 오류인지 구간별 부도율로 확인한다.

**도표 4A.** RevolvingUtilizationOfUnsecuredLines 구간별 비율

In [ ]:
bins = [0, 0.5, 1, 1.5, 2, 3, 5, 10, 100, float("inf")]
group = pd.cut(df["RevolvingUtilizationOfUnsecuredLines"], bins=bins, include_lowest=True)
ratio = (group.value_counts(sort=False) / len(df) * 100).round(2)
display(ratio.to_frame("비율(%)"))

**도표 4B.** RevolvingUtilizationOfUnsecuredLines 구간별 부도율

In [ ]:
show_default_rate(df, group, "RevolvingUtilizationOfUnsecuredLines 구간별 부도율")

**결론**

| 조건 | 처리 | 추가 컬럼 |
| --- | --- | --- |
| 2 이하 | 그대로 | - |
| 2 초과 ~ 100 이하 | 2로 자르기 | - |
| 100 초과 | 중앙값으로 대체 | `is_revolving_outlier` |

- **1 ~ 2 (1.96%)**: 부도율이 약 40%로 가장 높다(도표 4A, 4B). 한도를 넘겨 쓴 실제 고위험 고객이다.
- **2 ~ 100 (0.1%)**: 부도율이 23~33%로 여전히 평균의 3~5배다(도표 4A, 4B). 오류가 아니라 실제 고위험 고객이다. 148건뿐이라 2로 대체하여 1~2 구간과 합쳐도 방향이 틀리지 않는다.
- **100 초과 (0.15%)**: 부도율이 4.9%로 평균보다 낮다(도표 4A, 4B). 한도를 100배 넘게 쓴 사람이 평범한 고객보다 안전할 수는 없으므로 오류로 본다. 중앙값으로 대체하여, 부도율이 낮은 우량 고객임을 반영한다.

## 5. MonthlyIncome

MonthlyIncome이 결측이거나 0, 1처럼 비정상적으로 작은 고객의 부도율을 정상 고객과 비교한다.

**도표 5A.** MonthlyIncome 0, 1의 건수와 비율

In [ ]:
income = df["MonthlyIncome"]
special = income[income.isin([0, 1])].value_counts().sort_index()
display(pd.DataFrame({
    "건수": special,
    "비율(%)": (special / len(df) * 100).round(2),
}))

**도표 5B.** MonthlyIncome 구간별 부도율

In [ ]:
labels = ["결측", "0", "1", "2 ~ 100", "101 ~ 200", "201 ~ 300", "301 ~ 400", "401 ~"]
income_group = pd.Series("401 ~", index=df.index)
income_group[income <= 400] = "301 ~ 400"
income_group[income <= 300] = "201 ~ 300"
income_group[income <= 200] = "101 ~ 200"
income_group[income <= 100] = "2 ~ 100"
income_group[income == 1] = "1"
income_group[income == 0] = "0"
income_group[income.isna()] = "결측"
income_group = pd.Categorical(income_group, categories=labels, ordered=True)
show_default_rate(df, income_group, "MonthlyIncome 구간별 부도율")

**결론**

| 조건 | 처리 | 추가 컬럼 |
| --- | --- | --- |
| 300 초과 | 그대로 | - |
| 0 ~ 300 | 중앙값으로 대체 | `is_income_invalid` |
| 결측 (빈 값) | 중앙값으로 대체 | `is_income_missing` |

- 0과 1에 값이 몰려 있다(도표 5A, 5B). 실제 소득이 아닌 특수 코드로 본다.
- 2 ~ 300도 0, 1처럼 부도율이 평균보다 낮고, 300을 넘으면 평균 수준이 된다(도표 5B). 그래서 0 ~ 300을 이상치로 보고 중앙값으로 대체한다.
- `is_income_invalid`는 "실제 소득으로 쓸 수 없어 중앙값으로 바꾼 값"을 표시한다. 특수 코드(0, 1)와 비현실적으로 작은 값(2 ~ 300)이 모두 이에 해당한다.
- 2 ~ 400은 371건으로 모집단이 적어, 부도 몇 건 차이로 부도율이 크게 흔들리는 통계적 한계가 있다(도표 5B).
- 결측 고객도 부도율(5.6%)이 정상 고객과 달라 표시 컬럼으로 남긴다(도표 5B).

## 6. DebtRatio

DebtRatio는 "월 부채 상환액 ÷ MonthlyIncome"이다. MonthlyIncome 구간에 따라 DebtRatio와 부도율이 어떻게 달라지는지 확인한다.

**도표 6A.** MonthlyIncome 구간별 DebtRatio와 부도율

In [ ]:
income = df["MonthlyIncome"]
labels = ["결측", "0", "1", "2 ~ 300", "300 초과"]
income_group = pd.Series("300 초과", index=df.index, name="MonthlyIncome")
income_group[income <= 300] = "2 ~ 300"
income_group[income == 1] = "1"
income_group[income == 0] = "0"
income_group[income.isna()] = "결측"
income_group = income_group.astype(pd.CategoricalDtype(labels, ordered=True))

grouped = df.groupby(income_group, observed=False)
pd.DataFrame({
    "건수":             grouped.size(),
    "DebtRatio 중앙값": grouped["DebtRatio"].median(),
    "DebtRatio 99.9%":  grouped["DebtRatio"].quantile(0.999),
    "DebtRatio 최댓값": grouped["DebtRatio"].max(),
    "부도율(%)":        grouped[TARGET].mean() * 100,
}).round(2)

**도표 6B.** MonthlyIncome 300 초과 고객의 DebtRatio 구간별 부도율

In [ ]:
# DebtRatio가 클수록 부도율이 높은지 MonthlyIncome 300 초과 고객으로 확인
normal = df[df["MonthlyIncome"] > 300]
bins = [0, 0.1, 0.3, 0.5, 0.7, 1, 2, 10, float("inf")]
group = pd.cut(normal["DebtRatio"], bins=bins, include_lowest=True)
show_default_rate(normal, group, "MonthlyIncome 300 초과 고객의 DebtRatio 구간별 부도율")

**결론**

| MonthlyIncome | DebtRatio |
| --- | --- |
| 300 초과 | 99.9% 분위수(약 7)로 자르기 |
| 0 ~ 300 또는 결측 | 중앙값으로 대체 |

- DebtRatio는 MonthlyIncome으로 나눈 값이라, MonthlyIncome 구간별로 나눠서 확인했다.
- MonthlyIncome 300 초과: DebtRatio에 극단값(최대 약 170)이 있어 99.9% 분위수(약 7)로 자른다(도표 6A).
- MonthlyIncome 0 ~ 300, 결측: DebtRatio가 커서 위험한 고객처럼 보이지만, 실제 부도율은 평균보다 낮아 어폐가 있다(도표 6A, 6B). 이에 DebtRatio를 중앙값으로 대체한다.

### 6-1. 월 부채 상환액 컬럼 `monthly_debt_payment`

MonthlyIncome 0, 1, 결측 고객의 DebtRatio가 월 부채 상환액(금액)인지 확인한다. MonthlyIncome 300 초과 고객의 월 부채 상환액(DebtRatio × MonthlyIncome)과 규모를 비교한다.

**도표 6C.** 월 부채 상환액 비교

In [ ]:
income = df["MonthlyIncome"]
pct = [0.5, 0.9, 0.99]
pd.DataFrame({
    "MonthlyIncome 300 초과: DebtRatio × MonthlyIncome": (df["DebtRatio"] * income)[income > 300].quantile(pct),
    "MonthlyIncome 0: DebtRatio":                        df.loc[income == 0, "DebtRatio"].quantile(pct),
    "MonthlyIncome 1: DebtRatio":                        df.loc[income == 1, "DebtRatio"].quantile(pct),
    "MonthlyIncome 결측: DebtRatio":                     df.loc[income.isna(), "DebtRatio"].quantile(pct),
}).T.round(1)

**도표 6D.** `monthly_debt_payment` 분포

In [ ]:
# 1 초과는 DebtRatio × MonthlyIncome, 0, 1, 결측은 DebtRatio 그대로
payment = (df["DebtRatio"] * income).where(income > 1, df["DebtRatio"])
payment.describe(percentiles=[0.5, 0.99, 0.999]).to_frame("monthly_debt_payment").round(1)

**결론**

| MonthlyIncome | `monthly_debt_payment` |
| --- | --- |
| 1 초과 | DebtRatio × MonthlyIncome |
| 0, 1, 결측 | DebtRatio 값 그대로 |
| 공통 | 99.9% 분위수(약 27,000)로 자르기 |

- MonthlyIncome 300 초과(정상) 고객으로 계산한 월 부채 상환액(DebtRatio × MonthlyIncome)이, MonthlyIncome 0, 1, 결측 고객의 DebtRatio와 규모가 비슷하다(도표 6C). 그래서 이 DebtRatio를 상환액으로 판단했다.
- 이 DebtRatio를 중앙값으로 바꾸면 금액 정보가 사라지므로, `monthly_debt_payment`를 만들어 따로 담는다.
- `monthly_debt_payment`에도 극단값(최대 약 48만)이 있어 99.9% 분위수로 자른다(도표 6D).

## 7. NumberOfDependents

NumberOfDependents 결측 3,924건(2.62%)을 어떻게 채울지, 결측 여부를 따로 표시해야 하는지 확인한다.

**도표 7A.** MonthlyIncome과 NumberOfDependents의 결측 교차표

In [ ]:
income_na = df["MonthlyIncome"].isna()
dep_na = df["NumberOfDependents"].isna()
display(pd.DataFrame({
    "NumberOfDependents 있음": [(~income_na & ~dep_na).sum(), (income_na & ~dep_na).sum()],
    "NumberOfDependents 결측": [(~income_na & dep_na).sum(), (income_na & dep_na).sum()],
}, index=["MonthlyIncome 있음", "MonthlyIncome 결측"]))

**도표 7B.** 결측 조합별 부도율

In [ ]:
labels = ["둘 다 있음", "MonthlyIncome만 결측", "둘 다 결측"]
missing_group = pd.Series(labels[0], index=df.index)
missing_group[income_na & ~dep_na] = labels[1]
missing_group[income_na & dep_na] = labels[2]
missing_group = pd.Categorical(missing_group, categories=labels, ordered=True)
show_default_rate(df, missing_group, "결측 조합별 부도율")

**도표 7C.** NumberOfDependents 대표값

In [ ]:
# 값이 있는 고객 기준
dependents = df["NumberOfDependents"].dropna()
pd.DataFrame({
    "평균":        dependents.mean(),
    "중앙값":      dependents.median(),
    "0인 비율(%)": (dependents == 0).mean() * 100,
}, index=["NumberOfDependents"]).round(2)

**결론**

| 조건 | 처리 | 추가 컬럼 |
| --- | --- | --- |
| NumberOfDependents 결측 | 중앙값(0)으로 대체 | `is_dependents_missing` |

- NumberOfDependents 결측 3,924건은 모두 MonthlyIncome도 결측이다(도표 7A).
- 그런데 MonthlyIncome 결측 고객 안에서도 NumberOfDependents까지 결측이면 부도율이 더 낮다(5.77% → 4.56%, 도표 7B). `is_income_missing`만으로는 이 차이를 구분할 수 없어 `is_dependents_missing`을 추가한다.
- NumberOfDependents는 정수이고 0에 몰려 있어(59.5%), 평균(0.76) 대신 중앙값(0)으로 채운다(도표 7C).

## 8. 연체 횟수 컬럼 3개

`NumberOfTime30-59DaysPastDueNotWorse`, `NumberOfTime60-89DaysPastDueNotWorse`, `NumberOfTimes90DaysLate`에 있는 96, 98이 실제 연체 횟수인지, 어떤 고객인지 확인한다.

**도표 8A.** 연체 횟수 컬럼별 중앙값과 최댓값

In [ ]:
late = df[LATE_COLS]
display(pd.DataFrame({
    "중앙값":            late.median(),
    "0인 비율(%)":       ((late == 0).mean() * 100).round(2),
    "96/98 제외 최댓값": late.mask(late.isin([96, 98])).max().astype(int),
}))

**도표 8B.** 96/98 고객의 연체 횟수 컬럼 값 조합

In [ ]:
code = late.isin([96, 98]).any(axis=1)
display(df.loc[code, LATE_COLS].value_counts().reset_index(name="건수"))

**도표 8C.** 96/98 고객과 나머지 고객 비교

In [ ]:
def summarize(data: pd.DataFrame) -> pd.Series:
    """96/98 고객 비교표의 한 열(건수, 부도율, 주요 컬럼 통계)을 만든다."""
    return pd.Series({
        "건수":                                   len(data),
        "부도율(%)":                              data[TARGET].mean() * 100,
        "NumberOfOpenCreditLinesAndLoans 중앙값": data["NumberOfOpenCreditLinesAndLoans"].median(),
        "NumberRealEstateLoansOrLines 중앙값":    data["NumberRealEstateLoansOrLines"].median(),
        "age 중앙값":                             data["age"].median(),
        "MonthlyIncome 결측률(%)":                data["MonthlyIncome"].isna().mean() * 100,
    })

pd.DataFrame({
    "96/98 고객":  summarize(df[code]),
    "나머지 고객": summarize(df[~code]),
}).round(2)

**결론**

| 조건 | 처리 | 추가 컬럼 |
| --- | --- | --- |
| 연체 컬럼 3개가 96 또는 98 | 0으로 대체 | `is_late_unknown` |

- 연체 컬럼 3개가 항상 동시에 98(264명) 또는 96(5명)이다(도표 8B). 96/98을 빼면 최댓값이 17이므로, 실제 연체 횟수가 아닌 특수 코드로 본다(도표 8A).
- 96/98 고객은 사용 중인 계좌 0개, age 중앙값 29세, MonthlyIncome 결측 45%로 씬파일러의 특징을 보이고, 부도율은 약 55%로 추심 중인 고객의 특징을 보인다(도표 8C). 두 종류가 섞여 있는 것으로 본다.
- 그대로 두면 모델이 "98번 연체"로 읽어 스케일링과 SHAP 설명이 왜곡된다. 그래서 세 컬럼의 중앙값인 0(도표 8A)으로 바꾸고, 위험 신호는 `is_late_unknown`으로 남긴다.
- 부도 고객 147명(전체 부도 고객의 1.5%)이 들어 있어 행은 삭제하지 않는다(도표 8C).
- 두 종류를 이 데이터로 구분할 수 없어, `is_late_unknown`은 씬파일러 판정에 쓰지 않는다.

## 9. NumberRealEstateLoansOrLines

NumberRealEstateLoansOrLines가 20 이상인 고객 12명이 실제 값인지 확인한다. NumberRealEstateLoansOrLines는 NumberOfOpenCreditLinesAndLoans에 포함되므로, 더 크면 오류다.

**도표 9A.** NumberOfOpenCreditLinesAndLoans와 NumberRealEstateLoansOrLines 산점도

In [ ]:
# 두 값이 같은 선(점선)보다 위에 있는 점은 오류
estate = df["NumberRealEstateLoansOrLines"]
open_lines = df["NumberOfOpenCreditLinesAndLoans"]
many = df[estate >= 20]
pairs = df[["NumberOfOpenCreditLinesAndLoans", "NumberRealEstateLoansOrLines"]].drop_duplicates()

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(pairs["NumberOfOpenCreditLinesAndLoans"], pairs["NumberRealEstateLoansOrLines"],
           s=10, color="lightgray", label="전체 고객 (값 조합)")
ax.scatter(many["NumberOfOpenCreditLinesAndLoans"], many["NumberRealEstateLoansOrLines"],
           s=30, color="red", label=f"NumberRealEstateLoansOrLines 20 이상 ({len(many)}명)")
limit = max(open_lines.max(), estate.max()) + 2
ax.plot([0, limit], [0, limit], color="black", linestyle="--", linewidth=1, label="두 값이 같은 선")
ax.set_xlim(0, limit)
ax.set_ylim(0, limit)
ax.set_xlabel("NumberOfOpenCreditLinesAndLoans")
ax.set_ylabel("NumberRealEstateLoansOrLines")
ax.set_title(f"점선 위의 점 (NumberRealEstateLoansOrLines가 더 큰 행): {(estate > open_lines).sum()}건")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

**도표 9B.** NumberRealEstateLoansOrLines 20 이상 고객과 전체 고객 비교

In [ ]:
top10 = df["MonthlyIncome"].quantile(0.9)


def income_profile(data: pd.DataFrame) -> pd.Series:
    """건수, MonthlyIncome 중앙값, MonthlyIncome 상위 10%에 드는 비율(값이 있는 고객 기준)을 만든다."""
    known = data["MonthlyIncome"].dropna()
    return pd.Series({
        "건수":                                  len(data),
        "MonthlyIncome 중앙값":                  known.median(),
        "MonthlyIncome 상위 10%에 드는 비율(%)": (known > top10).mean() * 100,
    })


pd.DataFrame({
    "20 이상 고객": income_profile(many),
    "전체 고객":    income_profile(df),
}).round(2)

**결론: 그대로 둔다.**

- NumberRealEstateLoansOrLines가 NumberOfOpenCreditLinesAndLoans보다 큰 행은 0건이다(도표 9A). 값끼리 모순이 없다.
- NumberRealEstateLoansOrLines 20 이상인 고객은 MonthlyIncome 중앙값이 33,334로 전체(5,400)의 약 6배이고, MonthlyIncome이 있는 11명 중 10명이 상위 10%다(도표 9B). 부동산 투자자로 보는 것이 자연스럽다.

## 10. 최종 정리

| 컬럼 | 조건 | 처리 | 추가 컬럼 | 근거 |
| --- | --- | --- | --- | --- |
| `age` | 0 | 행 삭제 | - | 도표 3A |
| `RevolvingUtilizationOfUnsecuredLines` | 2 이하 | 그대로 | - | 도표 4A, 4B |
| | 2 초과 ~ 100 이하 | 2로 자르기 | - | |
| | 100 초과 | 중앙값으로 대체 | `is_revolving_outlier` | |
| `MonthlyIncome` | 300 초과 | 그대로 | - | 도표 5A, 5B |
| | 0 ~ 300 | 중앙값으로 대체 | `is_income_invalid` | |
| | 결측 | 중앙값으로 대체 | `is_income_missing` | |
| `DebtRatio` | MonthlyIncome 300 초과 | 99.9% 분위수(약 7)로 자르기 | - | 도표 6A, 6B |
| | MonthlyIncome 0 ~ 300 또는 결측 | 중앙값으로 대체 | - (`is_income_invalid`, `is_income_missing`이 표시) | |
| `monthly_debt_payment` (새 컬럼) | MonthlyIncome 1 초과 | DebtRatio × MonthlyIncome | - | 도표 6C, 6D |
| | MonthlyIncome 0, 1, 결측 | DebtRatio 값 그대로 | - | |
| | 공통 | 99.9% 분위수(약 27,000)로 자르기 | - | |
| `NumberOfDependents` | 결측 | 중앙값(0)으로 대체 | `is_dependents_missing` | 도표 7A ~ 7C |
| 연체 횟수 컬럼 3개 | 96 또는 98 | 0으로 대체 | `is_late_unknown` | 도표 8A ~ 8C |
| `NumberRealEstateLoansOrLines` | - | 그대로 | - | 도표 9A, 9B |
| `NumberOfOpenCreditLinesAndLoans` | - | 그대로 | - | 2번 값 분포 |

연체 횟수 컬럼 3개: `NumberOfTime30-59DaysPastDueNotWorse`, `NumberOfTime60-89DaysPastDueNotWorse`, `NumberOfTimes90DaysLate`

**처리 순서**
1. `age = 0` 행 1건을 삭제한다. Train/Validation/Test로 나누기 전에 하며, 삭제하는 행은 이것뿐이다.
2. `monthly_debt_payment`를 원래 값으로 계산한다. DebtRatio를 바꾸기 전에 해야 한다.
3. 표시 컬럼 5개를 만든다. 값을 바꾸기 전에 해야 한다.
4. 연체 횟수 컬럼의 96/98을 0으로, RevolvingUtilizationOfUnsecuredLines 2 초과 ~ 100 이하를 2로 바꾼다.
5. Train으로 계산한 중앙값과 99.9% 분위수로 채우고 자른다. Validation/Test에도 같은 값을 쓴다.
   - 중앙값: RevolvingUtilizationOfUnsecuredLines는 100 이하인 행, MonthlyIncome과 DebtRatio는 MonthlyIncome 300 초과인 행으로 계산
   - 99.9% 분위수: DebtRatio는 MonthlyIncome 300 초과인 행, monthly_debt_payment는 전체 행으로 계산